In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
from tqdm.auto import tqdm

from env.grid_world import Environment, State, Action, Reward, N_ACTIONS

from dataclasses import dataclass

In [ ]:
rng = np.random.default_rng(seed=0)

@dataclass
class Parameters:
  alpha: float  # step size
  epsilon: float  # e-greedy
  gamma: float  # discount factor


def select_action(q_table: np.ndarray, state: State, epsilon: float) -> Action:
  # using e-greedy method

  q: np.ndarray = q_table[*state]
  assert q.shape == (N_ACTIONS,)

  # exploratory
  if rng.random() < epsilon:
    a = rng.integers(0, N_ACTIONS)
  else:
    a = int(rng.choice(np.flatnonzero(q == q.max())))  # break ties randomly

  return Action(a)


In [ ]:
def update(
  q_table: np.ndarray,
  state: State,
  new_state: State,
  action: Action,
  reward: Reward,
  done: bool,
  params: Parameters,
):
  q_next = (1 - done) * np.max(q_table[*new_state])
  td_error = reward + params.gamma * q_next - q_table[*state, action.value]
  q_table[*state, action.value] += params.alpha * td_error


In [ ]:
env = Environment(dim=(8, 8))

# init q table
params = Parameters(alpha=0.1, epsilon=0.1, gamma=0.99)
num_episodes = 1000
max_steps = 5000

In [ ]:
print("starting training")

q_table = np.zeros((*env.dim, 4))

rewards = []

pbar_ep = tqdm(range(num_episodes), desc="Episodes")
for ep in pbar_ep:
  env.reset()
  total_reward = 0

  for step in range(max_steps):  # loop until terminal state
    action = select_action(q_table, env.pos, params.epsilon)
    reward, done = env.step(action)
    update(q_table, env.prev_pos, env.pos, action, reward, done, params)

    total_reward += reward

    if done:
      break
    
  rewards.append(total_reward)
  pbar_ep.set_postfix(steps=step + 1, reward=total_reward, eps=params.epsilon)


starting training


Episodes:   0%|          | 0/1000 [00:00<?, ?it/s]

In [5]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
  y=rewards,
  mode='lines',
  name='Raw',
  line=dict(color='steelblue')
))

fig.update_layout(
  title='Training Rewards',
  xaxis_title='Episode',
  yaxis_title='Total Reward',
)

fig.show()

In [ ]:
env.reset()

# test run
for step in range(max_steps):  # loop until terminal state
  env.render()
  
  action = select_action(q_table, env.pos, 0)
  reward, done = env.step(action)

  if done:
    break

A . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . G 

. A . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . G 

. . . . . . . . 
. A . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . G 

. . . . . . . . 
. . . . . . . . 
. A . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . G 

. . . . . . . . 
. . . . . . . . 
. . A . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . G 

. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . A . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . G 

. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . . 
. . A . . . . . 
. . . . . . . . 
. . . . . . . . 
. . . . . . . G 

. . . . . . . . 
. . . . . . . . 
. . . .